In [17]:
pip install mesa[rec]

In [18]:
# Has multi-dimensional arrays and matrices.
# Has a large collection of mathematical functions to operate on these arrays.
import numpy as np

# Data manipulation and analysis.
import pandas as pd

# Data visualization tools.
import seaborn as sns

import mesa

In [23]:
from mesa import Agent, Model
from mesa.time import RandomActivation
from mesa.datacollection import DataCollector
import matplotlib.pyplot as plt

# === Agent ===
class ClientAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        #self.unique_id = unique_id
        self.solvable = self.random.choice([True, False])
        self.loan_amount = self.random.randint(5000, 20000)

    def step(self):
        # Imaginons que le statut change parfois
        if self.random.random() < 0.05:
            self.solvable = not self.solvable

# === Modèle ===
class BankModel(Model):
    def __init__(self, N):
        self.num_agents = N
        self.schedule = RandomActivation(self)
        self.datacollector = DataCollector(
            model_reporters={
                "Solvables": lambda m: sum([1 for a in m.schedule.agents if a.solvable]),
                "Insolvables": lambda m: sum([1 for a in m.schedule.agents if not a.solvable])
            }
        )

        for i in range(self.num_agents):
            agent = ClientAgent(self)
            self.schedule.add(agent)

        self.datacollector.collect(self)  # collect initial state

    def step(self):
        self.schedule.step()
        self.datacollector.collect(self)

# === Simulation ===
model = BankModel(100)

for i in range(50):  # 50 étapes
    model.step()

# === Visualisation ===
data = model.datacollector.get_model_vars_dataframe()

plt.figure(figsize=(10, 5))
plt.plot(data["Solvables"], label="Solvables")
plt.plot(data["Insolvables"], label="Insolvables")
plt.xlabel("Étapes")
plt.ylabel("Nombre de clients")
plt.title("Évolution du nombre de clients solvables et insolvables")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


AttributeError: 'BankModel' object has no attribute '_agents'